# S3 — Fokker–Planck feasibility and λ selection (validation only; the TEST SET is not used)
All arms start from the **same** clean baseline (seed 42, epoch 40: weights + Adam state + loss scaler) and train **2 more
epochs** (→ epoch 42) with the exact Experiment-2 recipe and the same data order. The only difference between arms is λ.

| arm | λ | what |
|---|---|---|
| `s3_m0` | 0 | control: plain continuation of the baseline |
| `s3_l016` | 0.016 | FP term, small |
| `s3_l054` | 0.054 | FP term, medium |
| `s3_l107` | 0.107 | FP term, large |

FP term (once per optimizer step): `L_FP = r(u,v1)·r(u,v2) / S_ref(t)²`, t ∈ [10, 300], exact gradient by manual recompute,
added to the base gradient **before** the single clip at 1.0. Every step logs gradient norms, cos(base, FP), clipping and the
predicted change of both losses.

**Pre-stated decision** (`s3_decide.py`): R(λ) = 1 − H(λ)/H(M0) on 32 fixed validation images (32 patients), paired.
λ eligible if R ≥ 0.10, CI lower bound > 0, R_jvp > 0, validation loss and MSE ≤ 1.02× control, saturation ≤ 1.25× control
and ≤ 0.10, KID-mammo ≤ 1.5× control. Smallest eligible λ → LOCK. None eligible but some CI > 0 → INCONCLUSIVE (S3b).
Otherwise → H1 FAIL.

**How to run (on the account with the free quota)**
1. On the account that owns them, share the notebooks **clean-data-baseline-retraining** and the **S2 baseline-audit** notebook with
   this account (Share → add the account → *Can view*).
2. Here: Add Input → *Your Work* / *Shared with you* → **clean-data-baseline-retraining** (latest version) **and** the **S2 baseline-audit** notebook output.
3. Settings → Accelerator **GPU T4 ×2** · Internet **ON**.
4. **Save Version → Save & Run All (Commit)**. Then **stop the interactive session** (it also uses GPU quota).

**Time:** setup ≈ 15 min + smoke test ≈ 10 min + ≈ 4.9 h of the two GPU queues (FP arms ≈ 57 min/epoch, control ≈ 11 min/epoch, each evaluation ≈ 45 min incl. 200 samples) + report ≈ 2 min → **≈ 5.5 h of GPU quota**.
Hard limit `BUDGET_H = 10.5` h (Cell 1): a job is skipped if it would pass it; training stops cleanly at an epoch boundary.
**Send back:** `S3/S3_report.txt` + the `S3/grid_*.png` images (or the whole `S3_results.zip`).

In [ ]:
# ===================== CELL 1 — CONFIG =====================
BUDGET_H = 10.5           # hard limit for this session (12 h Kaggle limit - margin); the run itself needs ~5 h
ARMS = {"s3_m0": 0.0, "s3_l016": 0.016, "s3_l054": 0.054, "s3_l107": 0.107}   # control first
TO_EPOCH = 42             # warm start at epoch 40 -> 2 more epochs
RUN_SMOKE = True          # ~10 min end-to-end test of training + evaluation before the long jobs
import os, time
os.environ["PA_OUT"] = "/kaggle/working/S3"
os.makedirs(os.environ["PA_OUT"], exist_ok=True)
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
def used_h():
    return (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
print("S3 FP feasibility | budget", BUDGET_H, "h | arms", ARMS, "| output", os.environ["PA_OUT"])

In [ ]:
# ===================== CELL 2 — setup (needs Internet ON) =====================
import os, re, sys, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("/kaggle/working/U24_synthesis/.git"):
    subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
if not os.path.exists("/tmp/pip_done"):
    # Kaggle runs Python 3.13: install the repo requirements once, never force-reinstall numpy / pandas
    subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                   "pip install -q -r /tmp/req.txt", shell=True, check=True)
    subprocess.run("pip install -q --no-deps torch-fidelity", shell=True, check=True)
    open("/tmp/pip_done", "w").write("ok")
probe = ("import sys, numpy, pandas, scipy, torch, torchmetrics, monai, generative, PIL; "
         "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
         "'| torch', torch.__version__, '| torchmetrics', torchmetrics.__version__, '| monai', monai.__version__)")
env_chk = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
if env_chk.returncode != 0:
    print(env_chk.stderr[-2500:])
    raise SystemExit("!! environment check failed (see the error above) -> send Claude this block")
print("environment OK:", env_chk.stdout.strip())
p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
s = open(p).read()
s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers is not installed
if s2 != s:
    open(p, "w").write(s2)
print("flash attention ->", "patched off" if s2 != s else "already off")
m = "/kaggle/working/U24_synthesis/models.py"          # oracle loader (used later in A6): hard-coded config path
s = open(m).read()
s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
if s2 != s:
    open(m, "w").write(s2)
print("Cell 2 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")

In [ ]:
# ===================== CELL 3 — code + clean data (reused) + input checks + job runner =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.chdir(WD)
FILES = {"phaseA_common.py": r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = os.environ.get("PA_OUT", "/kaggle/working/phaseA")
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def clean_ckpt(seed, epoch=40):
    """clean-data baseline (step 2 of the timeline): base_clean_s<seed>_w_ep<epoch>.pt"""
    h = find(f"base_clean_s{seed}_w_ep{epoch:03d}.pt")
    if not h:
        raise SystemExit(f"base_clean_s{seed}_w_ep{epoch:03d}.pt not found: Add Input -> Your Work -> the clean-baseline "
                         "notebook (its latest output version), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(f"/tmp/{split}_cache.npz")
    return d["x"], d["y"]


def load_clean_meta(split):
    """clean cache WITH patient ids and md5 - same order as /tmp/<split>_cache.npz (written by prepare_clean.py)"""
    d = np.load(f"/kaggle/working/clean_data/{split}_cache.npz")
    return d["x"], d["y"], d["patient"].astype(str), d["md5"].astype(str)


def val_mask(patients, frac=0.15, seed=0):
    """patient-level VALIDATION split of the clean TRAIN set (classifiers, gates, guidance choice only).
    The diffusion model itself was trained on all of train. Fixed seed -> identical in every script/session."""
    ups = sorted(set(patients))
    rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''',
         "fp_core.py": r'''# fp_core.py - the Fokker-Planck (score-FPE) residual and its exact "manual recompute" gradient.
# Shared by the Phase A-2 diagnostics and (later) the Phase B/C training, so both use byte-identical maths.
# VP-SDE of the DDPM (tau = t/T, beta(tau) = -d log abar / d tau), score s = -eps_hat / sqrt(1 - abar_t):
#     d_tau s = 1/2 beta grad_x( x.s + div s + |s|^2 )
# projected on a unit direction u, Hutchinson probe v (Rademacher) for d_u(div s):
#     r(u,v) = u.d_tau s - 1/2 beta [ u.s + (J_s u).(x + 2 s) + v.(d_u J_s) v ]
# all derivatives by central finite differences (per-pixel step p; h_u = 256 p for unit u on 256x256, h_v = p);
# d_tau s by +-k integer steps. Product of two independent probes = unbiased estimate of (u.R)^2.
# Verified in Phase A: unit test vs autograd (1e-13), analytic Gaussian score -> residual ~0, wrong score -> O(1).
import math
import torch

T_STEPS = 1000
DTAU = 1.0 / T_STEPS


def p_of_t(t):
    """finite-difference step per t (fixed before Phase B from the A4 accuracy table: <= 1 % error vs the exact JVP)"""
    return 1e-3 if t <= 20 else (3e-3 if t <= 50 else 1e-2)


def beta_eff(logab, t):
    return float(-(logab[t + 1] - logab[t - 1]) / (2 * DTAU))


def plan(x, t, u, v1, v2, p, k=1):
    hu, hv = 256.0 * p, p
    E = [("tp", x, t + k), ("tm", x, t - k), ("s0", x, t), ("up", x + hu * u, t), ("um", x - hu * u, t)]
    for j, v in ((1, v1), (2, v2)):
        E += [(f"pp{j}", x + hu * u + hv * v, t), (f"pm{j}", x + hu * u - hv * v, t),
              (f"mp{j}", x - hu * u + hv * v, t), (f"mm{j}", x - hu * u - hv * v, t)]
    return E


def residuals(s, x, u, v1, v2, p, k, beta):
    hu, hv = 256.0 * p, p
    A = (u * (s["tp"] - s["tm"])).sum() / (2 * k * DTAU)
    Ju = (s["up"] - s["um"]) / (2 * hu)
    base = (u * s["s0"]).sum() + (Ju * (x + 2 * s["s0"])).sum()
    out = []
    for j, v in ((1, v1), (2, v2)):
        D2 = (v * (s[f"pp{j}"] - s[f"pm{j}"] - s[f"mp{j}"] + s[f"mm{j}"])).sum() / (4 * hu * hv)
        out.append(A - 0.5 * beta * (base + D2))
    return out[0], out[1], A


def coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w=1.0):
    """dL/ds_k for L = w r1 r2 (shared evaluations: w (r1 + r2) dr/ds_k; probe-specific: w r_other dr_j/ds_k)"""
    hu, hv = 256.0 * p, p
    b = 0.5 * beta
    Ju = (s["up"] - s["um"]) / (2 * hu)
    shared = {"tp": u / (2 * k * DTAU), "tm": -u / (2 * k * DTAU), "s0": -b * (u + 2 * Ju),
              "up": -b * (x + 2 * s["s0"]) / (2 * hu), "um": b * (x + 2 * s["s0"]) / (2 * hu)}
    g = {key: w * (r1 + r2) * val for key, val in shared.items()}
    for j, v, other in ((1, v1, r2), (2, v2, r1)):
        c = b * v / (4 * hu * hv)
        g[f"pp{j}"], g[f"pm{j}"], g[f"mp{j}"], g[f"mm{j}"] = -w * other * c, w * other * c, w * other * c, -w * other * c
    return g


def evaluate_nograd(sfn, E):
    s = {}
    with torch.no_grad():
        for tt in sorted({e[2] for e in E}):
            keys = [e[0] for e in E if e[2] == tt]
            out = sfn(torch.cat([e[1] for e in E if e[2] == tt]), tt)
            for i, kk in enumerate(keys):
                s[kk] = out[i:i + 1]
    return s


def fp_loss_and_grad(sfn, x, t, u, v1, v2, p, k, beta, w=1.0, group=1, backward=True):
    """exact gradient of L = w r1 r2 accumulated into .grad (memory of ONE network evaluation). Returns (L, r1, r2, lhs)."""
    E = plan(x, t, u, v1, v2, p, k)
    s = evaluate_nograd(sfn, E)
    with torch.no_grad():
        r1, r2, A = residuals(s, x, u, v1, v2, p, k, beta)
        g = coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w)
    if backward:
        for tt in sorted({e[2] for e in E}):
            grp = [e for e in E if e[2] == tt]
            for i in range(0, len(grp), group):
                part = grp[i:i + group]
                out = sfn(torch.cat([e[1] for e in part]), tt)
                (torch.cat([g[e[0]] for e in part]) * out).sum().backward()
    return float(w * r1 * r2), float(r1), float(r2), float(A)


def plan_n(x, t, u, vs, p, k=1):
    """same as plan() but for any number of probes vs (list of tensors)"""
    hu, hv = 256.0 * p, p
    E = [("tp", x, t + k), ("tm", x, t - k), ("s0", x, t), ("up", x + hu * u, t), ("um", x - hu * u, t)]
    for j, v in enumerate(vs):
        E += [(f"pp{j}", x + hu * u + hv * v, t), (f"pm{j}", x + hu * u - hv * v, t),
              (f"mp{j}", x - hu * u + hv * v, t), (f"mm{j}", x - hu * u - hv * v, t)]
    return E


def residuals_n(s, x, u, vs, p, k, beta):
    """r(u, v_j) for every probe j (same formula as residuals()); returns (list of r_j, lhs, terms dict)"""
    hu, hv = 256.0 * p, p
    A = (u * (s["tp"] - s["tm"])).sum() / (2 * k * DTAU)
    Ju = (s["up"] - s["um"]) / (2 * hu)
    us, Jt = (u * s["s0"]).sum(), (Ju * (x + 2 * s["s0"])).sum()
    rs, d2s = [], []
    for j, v in enumerate(vs):
        D2 = (v * (s[f"pp{j}"] - s[f"pm{j}"] - s[f"mp{j}"] + s[f"mm{j}"])).sum() / (4 * hu * hv)
        rs.append(A - 0.5 * beta * (us + Jt + D2)); d2s.append(D2)
    return rs, A, {"us": 0.5 * beta * us, "Jt": 0.5 * beta * Jt, "D2": [0.5 * beta * d for d in d2s]}


def make_score_fn(unet, ac, ctx_value, amp=False, chunk=4):
    """score function s(x, t) of the network for one class label (context value); fp32 by default (A4: fp16 unusable).
    Batches larger than `chunk` are split (non-flash attention at 64x64 needs ~0.5 GB per image in fp32)."""
    def sfn(xb, tt):
        if xb.shape[0] > chunk:
            return torch.cat([sfn(xb[i:i + chunk], tt) for i in range(0, xb.shape[0], chunk)])
        c = torch.full((xb.shape[0], 1, 1), float(ctx_value), device=xb.device, dtype=xb.dtype)
        tb = torch.full((xb.shape[0],), int(tt), device=xb.device, dtype=torch.long)
        if amp:
            with torch.autocast("cuda", dtype=torch.float16):
                e = unet(xb, timesteps=tb, context=c).float()
        else:
            e = unet(xb, timesteps=tb, context=c)
        return -e / math.sqrt(1.0 - float(ac[tt]))
    return sfn


def sref_interp(table, t):
    """log-linear interpolation of the reference scale table {t: RMS(u.d_tau s)} used to normalise the loss"""
    ts = sorted(int(k) for k in table)
    t = min(max(t, ts[0]), ts[-1])
    for a, b in zip(ts[:-1], ts[1:]):
        if a <= t <= b:
            la, lb = math.log(table[str(a)] if str(a) in table else table[a]), math.log(table[str(b)] if str(b) in table else table[b])
            return math.exp(la + (lb - la) * (t - a) / max(b - a, 1))
    return table[str(ts[-1])] if str(ts[-1]) in table else table[ts[-1]]
''',
         "a6_classifier.py": r'''# a6_classifier.py - A6: real-data benign/malignant classifier on the CLEAN split (+ shared training helpers).
# Used as (i) the "real-only" reference of the later downstream test, (ii) the judge of class conditioning (A7/A9),
# (iii) the "mammography feature" extractor for KID-mammo.
# PRE-STATED RECIPE (no tuning, no epoch selection): torchvision ConvNeXt-Tiny with ImageNet-1k weights,
# 256 px grayscale -> 3 channels, per-image min-max, ImageNet normalisation, AdamW lr 1e-4 wd 0.05, batch 32,
# 15 epochs, cosine schedule with 1 warm-up epoch, horizontal flip + small affine, class-balanced cross-entropy,
# fp16 autocast. FINAL epoch is used. Seeds 0, 1, 2. Trained on clean TRAIN minus the patient-level VALIDATION split.
# PRE-STATED RULE (validation only - the test set is never used for a decision):
#   usable as judge if the mean VALIDATION AUC over the 3 seeds is >= 0.65.  Test AUC is reported, not used.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, OUT

TAG = "A6"
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def prep(x01):
    """(B,1,H,W) any range -> per-image min-max -> 3ch ImageNet-normalised"""
    mn = x01.amin(dim=(1, 2, 3), keepdim=True); mx = x01.amax(dim=(1, 2, 3), keepdim=True)
    x = (x01 - mn) / (mx - mn + 1e-8)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD


def build():
    import torchvision
    m = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m


def auc(scores, labels):
    """Mann-Whitney AUC with tie handling (no sklearn dependency)"""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    order = np.argsort(s); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):
        m = s == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


@torch.no_grad()
def predict(m, X_u8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.as_tensor(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def augment(x, g):
    B = x.shape[0]
    u = lambda: torch.rand(B, generator=g, device=dev) * 2 - 1
    sx, sy, shx, shy = 1 + 0.05 * u(), 1 + 0.05 * u(), 0.05 * u(), 0.05 * u()
    z = torch.zeros(B, device=dev)
    th = torch.stack([torch.stack([sx, shx, z], 1), torch.stack([shy, sy, z], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=g, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)


def train_clf(X_u8, y, seed, epochs=15, bs=32, tag=TAG):
    """the fixed recipe above; returns the model after the FINAL epoch"""
    t0 = time.time()
    Xg, Yg = torch.from_numpy(X_u8).to(dev), torch.from_numpy(np.asarray(y)).long().to(dev)
    w = torch.tensor([len(y) / (2 * max(1, (np.asarray(y) == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed)
    m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(y) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev)
        tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss)
        if (ep + 1) % 5 == 0 or ep == 0:
            log(tag, f"seed {seed} epoch {ep + 1}/{epochs} train loss {tl / spe:.4f} | {(time.time() - t0) / 60:.1f} min")
    del Xg, Yg
    return m


def boot_auc(p, y, n=1000, seed=0):
    r = np.random.default_rng(seed); vals = []
    i0, i1 = np.where(np.asarray(y) == 0)[0], np.where(np.asarray(y) == 1)[0]
    for _ in range(n):
        b = np.concatenate([r.choice(i0, len(i0)), r.choice(i1, len(i1))])
        vals.append(auc(p[b], np.asarray(y)[b]))
    return [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=15)
    a = ap.parse_args()
    x, y, pat, _ = load_clean_meta("train"); xte, yte = load_cache("test")
    vm = val_mask(pat)
    res = {"recipe": "convnext_tiny IMAGENET1K_V1, AdamW 1e-4 wd .05, bs 32, 15 ep cosine (1 warm-up), flip+affine, "
                     "balanced CE, final epoch; trained on train minus validation patients",
           "n_train_fit": int((~vm).sum()), "n_val": int(vm.sum()), "n_val_patients": len(set(pat[vm])),
           "n_test": int(len(yte)), "seeds": {}}
    log(TAG, f"fit on {res['n_train_fit']} images | validation {res['n_val']} images ({res['n_val_patients']} patients)")
    pv_sum = np.zeros(int(vm.sum()))
    for seed in [int(s) for s in a.seeds.split(",")]:
        t0 = time.time()
        m = train_clf(x[~vm], y[~vm], seed, a.epochs)
        pv, pt = predict(m, x[vm]), predict(m, xte)
        pv_sum += pv
        res["seeds"][str(seed)] = {"val_auc": auc(pv, y[vm]), "test_auc_reported_only": auc(pt, yte),
                                   "val_balanced_acc": float(np.mean([((pv >= 0.5) == (y[vm] == 1))[y[vm] == c].mean() for c in (0, 1)])),
                                   "minutes": (time.time() - t0) / 60}
        torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, f"a6_clf_s{seed}.pt"))
        log(TAG, f"seed {seed}: VALIDATION AUC {res['seeds'][str(seed)]['val_auc']:.3f} | test AUC (reported only) "
                 f"{res['seeds'][str(seed)]['test_auc_reported_only']:.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
    va = [v["val_auc"] for v in res["seeds"].values()]
    res["mean_val_auc"] = float(np.mean(va)); res["sd_val_auc"] = float(np.std(va, ddof=1)) if len(va) > 1 else 0.0
    pe = pv_sum / len(va)
    res["ensemble_val_auc"] = auc(pe, y[vm]); res["ensemble_val_auc_ci95"] = boot_auc(pe, y[vm])
    res["mean_test_auc_reported_only"] = float(np.mean([v["test_auc_reported_only"] for v in res["seeds"].values()]))
    res["usable_as_judge"] = bool(res["mean_val_auc"] >= 0.65)
    save_json(res, "A6_classifier.json")
    log(TAG, f"mean VALIDATION AUC {res['mean_val_auc']:.3f} +- {res['sd_val_auc']:.3f} (ensemble {res['ensemble_val_auc']:.3f}) "
             f"-> usable as judge: {res['usable_as_judge']}")
''',
         "train_fp.py": r'''# train_fp.py - S3 (and later Phase C): continue training a clean baseline WITH (lam > 0) or WITHOUT (lam = 0) the
# Fokker-Planck consistency term. The BASE recipe is EXACTLY train_arm2.py (Experiment 2): 0.7 MSE + 0.2 (1-MS-SSIM)
# + 0.05 HF on x0_hat, per-image min-max to [-1,1], shear/scale/flip augmentation, 15 % label dropout (-1),
# batch 2 x grad-accum 4, Adam 1e-5, fp16 autocast + GradScaler, global grad-norm clip 1.0, no EMA, same data order
# (seed*1000+epoch). The physics-NLF hinge of Experiment 2 is NOT used (lam there = 0 for the baseline anyway).
# FP term (only if lam > 0), once per optimizer step, on image 0 of the LAST micro-batch of the accumulation group,
# with that image's training label (or -1 if it was dropped):
#   t ~ U{tmin..tmax}, x_t = sqrt(abar) x0 + sqrt(1-abar) eps, one unit direction u, two Rademacher probes v1, v2,
#   L_FP = r(u,v1) r(u,v2) / S_ref(t)^2 (fp_core: fp32 finite differences, fixed step table p(t)),
#   exact gradient by manual recompute (13 network evaluations). The network is in eval() mode for the FP evaluations
#   (no dropout / batch-norm in this UNet, so this only guarantees the function is deterministic).
#   Order: base grads unscaled (GradScaler.unscale_) -> + lam * grad L_FP -> ONE global clip at 1.0 -> Adam step.
#   All FP random draws come from their own generators (fp_seed*1000+epoch), identical in every arm, so the base
#   data / noise / dropout sequence is identical in every arm of a pair (paired design).
# Logged at every optimizer step (jsonl): base / weighted-FP / combined gradient norm before clipping, clip factor,
# cos(base, FP), the norm of the exact Adam update, and the first-order predicted change of the normalised FP loss
# and of the base (batch) loss along that update - computed from Adam's own update formula (no model copy).
import os, sys, glob, time, json, math, argparse
import numpy as np
import torch
import torch.nn.functional as F
from torchmetrics.functional.image import multiscale_structural_similarity_index_measure as msssim

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
sys.path.insert(0, WD)
import fp_core as fc

ap = argparse.ArgumentParser()
ap.add_argument("--arm", required=True)
ap.add_argument("--lam", type=float, default=0.0)
ap.add_argument("--epochs", type=int, required=True)     # TOTAL epoch number to reach (warm start continues the numbering)
ap.add_argument("--init", required=True)                 # <baseline>_last.pt (weights + optimizer + scaler) for a fresh arm
ap.add_argument("--sref_json", required=True)            # A8_fp_clean.json from S2 (S_ref table)
ap.add_argument("--seed", type=int, default=42)          # data order / noise seed - identical for all arms of a pair
ap.add_argument("--fp_seed", type=int, default=7919)
ap.add_argument("--tmin", type=int, default=10)
ap.add_argument("--tmax", type=int, default=300)
ap.add_argument("--bs", type=int, default=2)
ap.add_argument("--accum", type=int, default=4)
ap.add_argument("--lr", type=float, default=1e-5)
ap.add_argument("--p_uncond", type=float, default=0.15)
ap.add_argument("--budget_h", type=float, default=10.5)
ap.add_argument("--log_every", type=int, default=200)
ap.add_argument("--smoke_batches", type=int, default=0)
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
a = ap.parse_args()
TAG = f"[{a.arm}]"
def log(*m):
    print(TAG, *m, flush=True)
try:
    T0 = float(open("/tmp/pg_session_start").read())
except Exception:
    T0 = time.time()
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
os.makedirs(a.ckpt_dir, exist_ok=True)

def save_atomic(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    with open(tmp, "rb") as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, path)

def minmax_pm1(x):
    mn = x.amin(dim=(1, 2, 3), keepdim=True); mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1

_sob = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
KX = (_sob[None, None] / 8).to(dev); KY = (_sob.t()[None, None] / 8).to(dev)
def hf_loss(x, y):
    gx = F.conv2d(x, KX, padding=1) - F.conv2d(y, KX, padding=1)
    gy = F.conv2d(x, KY, padding=1) - F.conv2d(y, KY, padding=1)
    return (gx.pow(2) + gy.pow(2)).mean()

d = np.load("/tmp/train_cache.npz")
X = torch.from_numpy(d["x"]); Y = torch.from_numpy(d["y"]).long(); N = X.shape[0]
Xg, Yg = X.to(dev), Y.to(dev)
log(f"data {N} images {tuple(X.shape[1:])} | lam={a.lam} | FP t in [{a.tmin},{a.tmax}], once per optimizer step | "
    f"{torch.cuda.get_device_name(0)}")

def augment(img, gg):
    B = img.shape[0]
    do = (torch.rand(B, generator=gg, device=dev) < 0.9).float()
    u = lambda: (torch.rand(B, generator=gg, device=dev) * 2 - 1)
    sx, sy, shx, shy = 1 + 0.05 * u() * do, 1 + 0.05 * u() * do, 0.1 * u() * do, 0.1 * u() * do
    zero = torch.zeros(B, device=dev)
    theta = torch.stack([torch.stack([sx, shx, zero], 1), torch.stack([shy, sy, zero], 1)], 1)
    grid = F.affine_grid(theta, list(img.shape), align_corners=False)
    out = F.grid_sample(img, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=gg, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), out.flip(-1), out)

from model_architectures import MonaiDDPM
_m = MonaiDDPM(lr=a.lr, T=1000)
unet = _m.unet.to(dev)
ac = _m.scheduler.alphas_cumprod.detach().float().to(dev)
LOGAB = torch.log(_m.scheduler.alphas_cumprod.detach().double().cpu())
del _m
T = ac.numel()
opt = torch.optim.Adam(unet.parameters(), lr=a.lr)
scaler = torch.amp.GradScaler("cuda")
params = [p for p in unet.parameters()]
SREF = json.load(open(a.sref_json))["S_ref_table"]
log("S_ref table: " + ", ".join(f"{k}: {float(v):.3g}" for k, v in SREF.items()))

# ---------------- resume this arm if a checkpoint exists, else warm start from the baseline ----------------
own = [os.path.join(a.ckpt_dir, f"{a.arm}_last.pt")]
if not a.smoke_batches:
    own += glob.glob(f"/kaggle/input/**/{a.arm}_last.pt", recursive=True)
cands = []
for p in own:
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
            cands.append((int(ck["step"]), p)); del ck
        except Exception as e:
            log("unreadable checkpoint (skipped):", p, repr(e)[:150])
src = max(cands)[1] if cands else a.init
ck = torch.load(src, map_location="cpu", weights_only=False)
if cands:
    for k in ("lam", "lr", "bs", "accum", "seed", "p_uncond", "fp_seed", "tmin", "tmax"):
        if ck["args"].get(k) != getattr(a, k):
            log(f"!! WARNING {k} differs: checkpoint={ck['args'].get(k)} now={getattr(a, k)}")
unet.load_state_dict(ck["unet"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
start_epoch, step, opt_steps = int(ck["epoch"]), int(ck["step"]), int(ck["opt_steps"])
history = ck.get("history", []) if cands else []
init_epoch = int(ck.get("init_epoch", ck["epoch"])) if cands else int(ck["epoch"])
del ck; torch.cuda.empty_cache()
n_opt = sum(1 for s in opt.state.values() if s)
chk = float(sum(p.detach().float().sum() for p in params[:5]))
log(f"START epoch={start_epoch} step={step} opt_steps={opt_steps} opt_states={n_opt} lam={a.lam} checksum={chk:.6f} "
    f"init_epoch={init_epoch} from={src}")
if n_opt == 0:
    log("!! WARNING: optimizer state is EMPTY -> this is not a true continuation of the baseline")

def sqnorm(gs):
    return sum(float((g.float() ** 2).sum()) for g in gs if g is not None)

@torch.no_grad()
def adam_update_stats(g_base, g_fp):
    """exact torch.optim.Adam update (no weight decay / amsgrad) for the CURRENT (clipped) .grad, tensor by tensor:
    returns |update|, g_base . update, g_fp . update (first-order predicted change of each loss)"""
    gr = opt.param_groups[0]; b1, b2 = gr["betas"]; lr, eps = gr["lr"], gr["eps"]
    nrm2, d_base, d_fp = 0.0, 0.0, 0.0
    for j, p in enumerate(params):
        g = p.grad
        if g is None:
            continue
        st = opt.state.get(p, {})
        k = float(st["step"]) + 1 if "step" in st else 1.0
        m = b1 * st["exp_avg"] + (1 - b1) * g if "exp_avg" in st else (1 - b1) * g
        v = b2 * st["exp_avg_sq"] + (1 - b2) * g * g if "exp_avg_sq" in st else (1 - b2) * g * g
        upd = -(lr / (1 - b1 ** k)) * m / (v.sqrt() / math.sqrt(1 - b2 ** k) + eps)
        nrm2 += float((upd * upd).sum())
        if g_base[j] is not None:
            d_base += float((g_base[j] * upd).sum())
        if g_fp is not None and g_fp[j] is not None:
            d_fp += float((g_fp[j] * upd).sum())
    return math.sqrt(nrm2), d_base, d_fp

spe = N // a.bs
if a.smoke_batches:
    spe = min(spe, a.smoke_batches)
durs = [h["minutes"] * 60 for h in history[-3:]]
fplog = open(os.path.join(a.ckpt_dir, f"{a.arm}_fp_log.jsonl"), "a")
reason = "max epochs reached"
bad = 0
for epoch in range(start_epoch, a.epochs):
    used_h = (time.time() - T0) / 3600
    if durs and used_h + 1.15 * max(durs[-3:]) / 3600 + 0.2 > a.budget_h:
        reason = f"time budget ({used_h:.2f} h used of {a.budget_h} h)"; break
    gcpu = torch.Generator().manual_seed(a.seed * 1000 + epoch)
    perm = torch.randperm(N, generator=gcpu)
    gg = torch.Generator(device=dev).manual_seed(a.seed * 1000 + epoch)
    gfp = torch.Generator(device=dev).manual_seed(a.fp_seed * 1000 + epoch)          # FP draws: same in every arm
    rfp = np.random.default_rng(a.fp_seed * 1000 + epoch)
    unet.train(); opt.zero_grad(set_to_none=True)
    t_ep, t_win = time.time(), time.time()
    S = dict(loss=0.0, mse=0.0, ms=0.0, hf=0.0, fp=0.0, n_fp=0, fp_skipped=0, clipped=0, n_steps=0)
    n_ok = 0
    for i in range(spe):
        idx = perm[i * a.bs:(i + 1) * a.bs].to(dev)
        x0 = minmax_pm1(augment(Xg[idx].float().unsqueeze(1), gg)); y = Yg[idx]
        t = torch.randint(0, T, (a.bs,), generator=gg, device=dev)
        eps = torch.randn(x0.shape, generator=gg, device=dev)
        drop = torch.rand(a.bs, generator=gg, device=dev) < a.p_uncond
        cond = torch.where(drop, torch.full_like(y, -1), y).float().view(a.bs, 1, 1)
        ab = ac[t].view(-1, 1, 1, 1)
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
        with torch.autocast("cuda", dtype=torch.float16):
            eps_hat = unet(xt, timesteps=t, context=cond)
        eps_hat = eps_hat.float()
        mse = F.mse_loss(eps_hat, eps)
        x0_hat = ((xt - (1 - ab).sqrt() * eps_hat) / ab.sqrt()).clamp(-1, 1)
        ms = 1.0 - msssim((x0_hat + 1) / 2, (x0 + 1) / 2, data_range=2.0, gaussian_kernel=False, normalize="relu")
        hf = hf_loss(x0_hat, x0)
        loss = 0.7 * mse + 0.2 * ms + 0.05 * hf                     # exact original loss
        if not torch.isfinite(loss):                                 # same handling as train_arm2.py
            bad += 1
            if bad > 50:
                raise RuntimeError("50 consecutive non-finite losses - stopping")
            continue
        bad = 0
        scaler.scale(loss / a.accum).backward()
        step += 1
        if (i + 1) % a.accum == 0 or i == spe - 1:
            scaler.unscale_(opt)
            rec = {"epoch": epoch + 1, "opt_step": opt_steps + 1}
            g_base = [p.grad.detach().clone() if p.grad is not None else None for p in params]
            nb = math.sqrt(sqnorm(g_base)); rec["base_norm"] = nb
            g_fp = None
            if a.lam > 0:
                # ---- one FP sample: image 0 of this micro-batch, its own random draws ----
                tf = int(rfp.integers(a.tmin, a.tmax + 1))
                abf = float(ac[tf])
                xf0 = x0[0:1].detach()
                xf = math.sqrt(abf) * xf0 + math.sqrt(1 - abf) * torch.randn(xf0.shape, generator=gfp, device=dev)
                u = torch.randn(xf.shape, generator=gfp, device=dev); u = u / u.norm()
                v1 = (torch.randint(0, 2, xf.shape, generator=gfp, device=dev) * 2 - 1).float()
                v2 = (torch.randint(0, 2, xf.shape, generator=gfp, device=dev) * 2 - 1).float()
                lab = float(cond[0].item())
                for p in params:
                    p.grad = None
                w = a.lam / fc.sref_interp(SREF, tf) ** 2
                unet.eval()
                Lw, r1, r2, A = fc.fp_loss_and_grad(fc.make_score_fn(unet, ac, lab), xf, tf, u, v1, v2,
                                                    fc.p_of_t(tf), 1, fc.beta_eff(LOGAB, tf), w=w)
                unet.train()
                g_fp = [p.grad.detach().clone() if p.grad is not None else None for p in params]
                nf = math.sqrt(sqnorm(g_fp))
                ok = math.isfinite(Lw) and math.isfinite(nf)
                for p, gb, gf in zip(params, g_base, g_fp):
                    use_f = ok and gf is not None
                    p.grad = None if (gb is None and not use_f) else \
                        (gb + gf if (gb is not None and use_f) else (gb.clone() if gb is not None else gf.clone()))
                rec.update({"t": tf, "label": lab, "L_fp_norm": Lw / a.lam, "r1": r1, "r2": r2, "lhs": A,
                            "fp_norm_weighted": nf, "fp_ok": ok,
                            "cos_base_fp": (sum(float((x_ * y_).sum()) for x_, y_ in zip(g_base, g_fp)
                                                if x_ is not None and y_ is not None) / (nb * nf + 1e-30)) if ok else None})
                if ok:
                    S["fp"] += Lw / a.lam; S["n_fp"] += 1
                else:
                    S["fp_skipped"] += 1; g_fp = None
            comb = float(torch.nn.utils.clip_grad_norm_(params, 1.0))
            rec["combined_norm"] = comb; rec["clip_factor"] = min(1.0, 1.0 / (comb + 1e-6))
            S["clipped"] += int(comb > 1.0); S["n_steps"] += 1
            un, d_base, d_fp = adam_update_stats(g_base, g_fp)
            rec["update_norm"] = un
            rec["pred_dL_base"] = d_base                                   # base (batch) loss gradient . update
            if g_fp is not None:
                rec["pred_dL_fp_norm"] = d_fp / a.lam                      # normalised FP loss gradient . update
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            opt_steps += 1
            fplog.write(json.dumps(rec) + "\n")
            if opt_steps % 25 == 0:
                fplog.flush()
            del g_base, g_fp
        n_ok += 1
        for k_, v_ in (("loss", loss), ("mse", mse), ("ms", ms), ("hf", hf)):
            S[k_] += float(v_.detach())
        if (i + 1) % a.log_every == 0:
            ips = a.log_every / (time.time() - t_win); t_win = time.time()
            m = {k: S[k] / n_ok for k in ("loss", "mse", "ms", "hf")}
            log(f"ep {epoch + 1} it {i + 1}/{spe} step {step} loss {m['loss']:.4f} mse {m['mse']:.4f} ms {m['ms']:.4f} "
                f"hf {m['hf']:.4f} | clipped {S['clipped']}/{S['n_steps']} steps"
                + (f" | FP {S['fp'] / max(S['n_fp'], 1):+.4f} ({S['n_fp']} samples, {S['fp_skipped']} skipped)" if a.lam > 0 else "")
                + f" | {ips:.2f} it/s")
    dur = time.time() - t_ep; durs.append(dur)
    m = {k: S[k] / max(n_ok, 1) for k in ("loss", "mse", "ms", "hf")}
    m["fp_norm_mean"] = S["fp"] / S["n_fp"] if S["n_fp"] else None
    m.update({"fp_samples": S["n_fp"], "fp_skipped": S["fp_skipped"], "clipped_share": S["clipped"] / max(S["n_steps"], 1)})
    history.append({"epoch": epoch + 1, "step": step, "opt_steps": opt_steps, **m, "minutes": dur / 60})
    save_atomic({"unet": unet.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch + 1,
                 "step": step, "opt_steps": opt_steps, "args": vars(a), "history": history, "init_epoch": init_epoch},
                os.path.join(a.ckpt_dir, f"{a.arm}_last.pt"))
    snap = {"unet": {k: v.detach().half().cpu() for k, v in unet.state_dict().items()}, "epoch": epoch + 1, "step": step,
            "arm": a.arm, "lam": a.lam, "init_epoch": init_epoch}
    save_atomic(snap, os.path.join(a.ckpt_dir, f"{a.arm}_w_ep{epoch + 1:03d}.pt")); del snap
    with open(os.path.join(a.ckpt_dir, f"{a.arm}_history.json"), "w") as fh:
        json.dump(history, fh, indent=1)
    fplog.flush()
    log(f"SAVED epoch={epoch + 1} step={step} opt_steps={opt_steps} loss={m['loss']:.4f} mse={m['mse']:.4f} "
        + (f"FP={m['fp_norm_mean']:+.4f} " if m["fp_norm_mean"] is not None else "")
        + f"clipped {m['clipped_share']:.2f} | {dur / 60:.1f} min/epoch | peak GPU mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
fplog.close()
log(f"FINISHED epochs_completed={history[-1]['epoch'] if history else start_epoch} step={step} reason={reason}")
''',
         "s3_eval.py": r'''# s3_eval.py - S3 evaluation of one arm at one or more epochs (VALIDATION images only; the test set is not touched).
#  1. H1 PROBE SET (every epoch): 32 validation images from 32 different patients (16 benign / 16 malignant, fixed),
#     t in {10,20,50,100,200,300}, one unit direction u and n_v = 2 Rademacher probes per (image, t), all drawn once from
#     a fixed CPU seed -> identical x_t, u, v in every arm and epoch (paired). fp32 finite differences with the fixed
#     step table p(t). P(i,t) = r(u,v1) r(u,v2) / S_ref(t)^2 (unbiased for (u.R)^2 / S_ref^2), H(i) = mean_t P(i,t).
#  2. VALIDATION LOSS (every epoch): the training loss (0.7 MSE + 0.2 (1-MS-SSIM) + 0.05 HF) and the MSE on all 347
#     validation images x 4 fixed (t, noise) pairs, true labels, no augmentation, fp16 autocast as in training.
#  3. FINAL EPOCH only (--final):
#     a. exact-JVP check: 8 probe images (4/4) x t in {20, 200}, same x_t, u, v1, v2 -> P_jvp (decides "R_jvp > 0")
#        and the FD-vs-JVP relative error after training (FP training must not exploit the FD error).
#     b. samples: n_gen per class at guidance 2 (primary), DDIM 50, the S2 noise bank (seed 2025 + 100000*label + batch)
#        -> saturation share / brightness / HF energy (istats of A9), KID-mammo vs the train-fit images (A6 seed-0
#        ConvNeXt penultimate features, same kid() as A9), memorisation (64x64 pixel correlation to every train image vs
#        the validation->train-fit reference q99), 8-per-class grid png.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, val_mask, minmax_pm1, load_model, find, OUT
import fp_core as fc

ap = argparse.ArgumentParser()
ap.add_argument("--arm", required=True)
ap.add_argument("--epochs", required=True)               # e.g. "41,42"
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
ap.add_argument("--ckpt", default="")                    # explicit snapshot path (single epoch), e.g. the epoch-40 baseline
ap.add_argument("--final", action="store_true")          # run part 3 at the LAST listed epoch
ap.add_argument("--n_probe", type=int, default=32)
ap.add_argument("--n_v", type=int, default=2)
ap.add_argument("--grid", default="10,20,50,100,200,300")
ap.add_argument("--n_rep", type=int, default=4)
ap.add_argument("--n_gen", type=int, default=100)
ap.add_argument("--gs", type=float, default=2.0)
ap.add_argument("--steps", type=int, default=50)
ap.add_argument("--smoke", action="store_true")
a = ap.parse_args()
if a.smoke:
    a.n_probe, a.grid, a.n_rep, a.n_gen, a.steps = 2, "20,200", 1, 2, 5
TAG = f"S3-eval-{a.arm}"
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
dev = torch.device("cuda")
GRID = [int(v) for v in a.grid.split(",")]
EPOCHS = [int(v) for v in a.epochs.split(",")]
SREF = json.load(open(find("A8_fp_clean.json")[0]))["S_ref_table"]

# ---------------- fixed validation material (identical in every arm / epoch / session) ----------------
xtr, ytr, pat, _ = load_clean_meta("train")
vm = val_mask(pat)
XV, YV, PV = xtr[vm], ytr[vm], pat[vm]
XF = xtr[~vm]
rng = np.random.default_rng(0)
order = rng.permutation(len(YV))
pick, used, cnt = {0: [], 1: []}, set(), {0: 0, 1: 0}
for j in order:
    c = int(YV[j])
    if PV[j] in used or cnt[c] >= a.n_probe // 2:
        continue
    pick[c].append(int(j)); used.add(PV[j]); cnt[c] += 1
PIDX = np.array(pick[0] + pick[1])                       # first half benign, second half malignant
assert len(PIDX) == a.n_probe and len(set(PV[PIDX])) == a.n_probe
X0 = minmax_pm1(torch.from_numpy(XV[PIDX]).float().unsqueeze(1)).to(dev)
LAB = [int(v) for v in YV[PIDX]]
H, W = X0.shape[-2:]
gcpu = torch.Generator().manual_seed(20261020)
NOISE = torch.randn((a.n_probe, len(GRID), 1, H, W), generator=gcpu)
U = torch.randn((a.n_probe, len(GRID), 1, H, W), generator=gcpu)
U = U / U.flatten(2).norm(dim=2).view(a.n_probe, len(GRID), 1, 1, 1)
V = (torch.randint(0, 2, (a.n_probe, len(GRID), a.n_v, 1, H, W), generator=gcpu) * 2 - 1).float()
NV = 4 if a.smoke else len(YV)
rv = np.random.default_rng(123)
TV = rv.integers(0, 1000, size=(NV, a.n_rep))
gnv = torch.Generator().manual_seed(321)
EPSV = [torch.randn((NV, 1, H, W), generator=gnv) for _ in range(a.n_rep)]

from torchmetrics.functional.image import multiscale_structural_similarity_index_measure as msssim
_sob = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
KX, KY = (_sob[None, None] / 8).to(dev), (_sob.t()[None, None] / 8).to(dev)
def hf_per_image(x, y):
    gx = F.conv2d(x, KX, padding=1) - F.conv2d(y, KX, padding=1)
    gy = F.conv2d(x, KY, padding=1) - F.conv2d(y, KY, padding=1)
    return (gx.pow(2) + gy.pow(2)).mean((1, 2, 3))

def ms_per_image(p, q):
    """MS-SSIM per image with the training arguments (reduction='none'; per-image loop if this torchmetrics lacks it)"""
    try:
        return msssim(p, q, data_range=2.0, gaussian_kernel=False, normalize="relu", reduction="none").reshape(-1).float()
    except Exception:
        return torch.stack([msssim(p[j:j + 1], q[j:j + 1], data_range=2.0, gaussian_kernel=False, normalize="relu")
                            for j in range(p.shape[0])]).reshape(-1).float()

def probe_set(unet, ac, LOGAB):
    SF = {lab: fc.make_score_fn(unet, ac, lab) for lab in (0, 1)}
    R = np.zeros((a.n_probe, len(GRID), a.n_v)); A_ = np.zeros((a.n_probe, len(GRID)))
    t0 = time.time()
    for ti, t in enumerate(GRID):
        p, beta, ab = fc.p_of_t(t), fc.beta_eff(LOGAB, t), float(ac[t])
        for i in range(a.n_probe):
            x = math.sqrt(ab) * X0[i:i + 1] + math.sqrt(1 - ab) * NOISE[i, ti:ti + 1].to(dev)
            u = U[i, ti:ti + 1].to(dev); vs = [V[i, ti, q:q + 1].to(dev) for q in range(a.n_v)]
            s = fc.evaluate_nograd(SF[LAB[i]], fc.plan_n(x, t, u, vs, p))
            rs, A, _ = fc.residuals_n(s, x, u, vs, p, 1, beta)
            R[i, ti] = [float(r) for r in rs]; A_[i, ti] = float(A)
        log(TAG, f"  probe t={t:4d}: done ({(time.time() - t0) / 60:.1f} min)")
    sr = np.array([fc.sref_interp(SREF, t) for t in GRID])
    nv = a.n_v
    P = (R.sum(2) ** 2 - (R ** 2).sum(2)) / (nv * (nv - 1)) / sr[None, :] ** 2      # = r1 r2 / S_ref^2 for n_v = 2
    return R, A_, P

@torch.no_grad()
def val_loss(unet, ac):
    acf = ac.float(); comp = np.zeros((NV, a.n_rep)); mse = np.zeros((NV, a.n_rep))
    for k in range(a.n_rep):
        for i in range(0, NV, 8):
            x0 = minmax_pm1(torch.from_numpy(XV[i:min(i + 8, NV)]).float().unsqueeze(1).to(dev))
            n = x0.shape[0]
            t = torch.as_tensor(TV[i:i + n, k], device=dev, dtype=torch.long)
            eps = EPSV[k][i:i + n].to(dev)
            cond = torch.as_tensor(YV[i:i + n], device=dev).float().view(n, 1, 1)
            ab = acf[t].view(-1, 1, 1, 1)
            xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
            with torch.autocast("cuda", dtype=torch.float16):
                eh = unet(xt, timesteps=t, context=cond)
            eh = eh.float()
            m = ((eh - eps) ** 2).mean((1, 2, 3))
            x0h = ((xt - (1 - ab).sqrt() * eh) / ab.sqrt()).clamp(-1, 1)
            ms = 1.0 - ms_per_image((x0h + 1) / 2, (x0 + 1) / 2)
            c = 0.7 * m + 0.2 * ms + 0.05 * hf_per_image(x0h, x0)
            comp[i:i + n, k] = c.cpu().numpy(); mse[i:i + n, k] = m.cpu().numpy()
    return comp, mse

from torch.func import jvp
def r_jvp(x, t, sfn, u, v, LOGAB):
    beta = fc.beta_eff(LOGAB, t)
    with torch.no_grad():
        s0 = sfn(x, t); A = (u * (sfn(x, t + 1) - sfn(x, t - 1))).sum() / (2 * fc.DTAU)
    _, Ju = jvp(lambda z: sfn(z, t), (x,), (u,))
    def g(z):
        _, jv = jvp(lambda w: sfn(w, t), (z,), (v,))
        return (v * jv).sum()
    _, d2 = jvp(g, (x,), (u,))
    return float(A - 0.5 * beta * ((u * s0).sum() + (Ju * (x + 2 * s0)).sum() + d2))

def jvp_check(unet, ac, LOGAB, R_fd):
    SF = {lab: fc.make_score_fn(unet, ac, lab) for lab in (0, 1)}
    h = a.n_probe // 2; imgs = list(range(min(4, h))) + list(range(h, h + min(4, h)))
    out = {"images": imgs, "t": [], "P_jvp": [], "P_fd": [], "r_jvp": [], "r_fd": []}
    for t in (20, 200):
        if t not in GRID:
            continue
        ti = GRID.index(t); ab = float(ac[t]); sr2 = fc.sref_interp(SREF, t) ** 2
        for i in imgs:
            x = math.sqrt(ab) * X0[i:i + 1] + math.sqrt(1 - ab) * NOISE[i, ti:ti + 1].to(dev)
            u = U[i, ti:ti + 1].to(dev)
            rj = [r_jvp(x, t, SF[LAB[i]], u, V[i, ti, q:q + 1].to(dev), LOGAB) for q in range(2)]
            out["t"].append(t); out["r_jvp"].append(rj); out["r_fd"].append([float(R_fd[i, ti, q]) for q in range(2)])
            out["P_jvp"].append(rj[0] * rj[1] / sr2); out["P_fd"].append(float(R_fd[i, ti, 0] * R_fd[i, ti, 1]) / sr2)
    rj, rf = np.array(out["r_jvp"]).ravel(), np.array(out["r_fd"]).ravel()
    out["rel_err_fd_vs_jvp"] = float(np.sqrt(np.mean((rf - rj) ** 2)) / (np.sqrt(np.mean(rj ** 2)) + 1e-300))
    return out

# ---------------- part 3b helpers (copied from a9_metrics.py: same definitions) ----------------
def istats(X_u8):
    I = torch.from_numpy(X_u8).float().div(255).unsqueeze(1)
    mn = I.amin(dim=(1, 2, 3), keepdim=True); mx = I.amax(dim=(1, 2, 3), keepdim=True); I = (I - mn) / (mx - mn + 1e-8)
    mu = F.avg_pool2d(I, 7, 1, 3, count_include_pad=False); m = (mu > 0.04).float(); n = m.sum((1, 2, 3)).clamp_min(1)
    hp = I - F.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
    fr = torch.ones_like(I); fr[..., 4:-4, 4:-4] = 0
    return {"mean_breast": float(((I * m).sum((1, 2, 3)) / n).mean()),
            "saturation_share": float((((I > 0.95).float() * m).sum((1, 2, 3)) / n).mean()),
            "hf_energy": float(((hp * hp * m).sum((1, 2, 3)) / n).mean()),
            "breast_area": float(m.mean((1, 2, 3)).mean()), "border_brightness": float(((I * fr).sum((1, 2, 3)) / fr.sum((1, 2, 3))).mean())}

def kid(fa, fb, n_sub=100, size=100, seed=0):
    r = np.random.default_rng(seed); d = fa.shape[1]; vals = []; size = min(size, len(fa), len(fb))
    for _ in range(n_sub):
        A_ = fa[r.choice(len(fa), size, replace=False)]; B_ = fb[r.choice(len(fb), size, replace=False)]
        kxx = (A_ @ A_.T / d + 1) ** 3; kyy = (B_ @ B_.T / d + 1) ** 3; kxy = (A_ @ B_.T / d + 1) ** 3
        vals.append((kxx.sum() - np.trace(kxx)) / (size * (size - 1)) + (kyy.sum() - np.trace(kyy)) / (size * (size - 1)) - 2 * kxy.mean())
    return float(np.mean(vals)), float(np.std(vals))

def thumbs(X_u8):
    t = X_u8.reshape(len(X_u8), 64, 4, 64, 4).mean((2, 4)).reshape(len(X_u8), -1).astype(np.float64)
    t -= t.mean(1, keepdims=True); t /= np.linalg.norm(t, axis=1, keepdims=True) + 1e-9
    return t

@torch.no_grad()
def ddim(unet, acf, label, n, gen, gs):
    T = acf.numel()
    x = torch.randn(n, 1, 256, 256, generator=gen, device=dev)
    ts = torch.linspace(T - 1, 0, a.steps, device=dev).round().long()
    cond = torch.full((n, 1, 1), float(label), device=dev)
    for i, t in enumerate(ts):
        ab_t = acf[t]
        ab_prev = acf[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=dev)
        tb = torch.full((2 * n,), int(t), device=dev, dtype=torch.long)
        ctx = torch.cat([-torch.ones_like(cond), cond]).half()
        eu, ec = unet(torch.cat([x, x]).half(), timesteps=tb, context=ctx).float().chunk(2)
        eps = eu + gs * (ec - eu)
        x0 = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1, 1)
        eps = (x - ab_t.sqrt() * x0) / (1 - ab_t).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return ((x + 1) / 2).clamp(0, 1)

def final_quality(unet, ac, name):
    from PIL import Image
    from a6_classifier import build, prep
    acf = ac.float(); unet = unet.half(); t0 = time.time(); imgs, labels = [], []
    for label in (0, 1):
        done, b = 0, 0
        while done < a.n_gen:
            k = min(16, a.n_gen - done)
            out = ddim(unet, acf, label, k, torch.Generator(device=dev).manual_seed(2025 + 100000 * label + b), a.gs)[:, 0].cpu()
            imgs.append((out.numpy() * 255).round().astype(np.uint8)); labels += [label] * k; done += k; b += 1
    G = np.concatenate(imgs); L = np.array(labels, dtype=np.int64)
    np.savez_compressed(os.path.join(OUT, f"gen_{name}_gs{a.gs:g}.npz"), x=G, y=L)
    kk = min(8, a.n_gen)
    Image.fromarray(np.concatenate([np.concatenate([G[j + c * a.n_gen] for j in range(kk)], 1) for c in (0, 1)], 0)).save(
        os.path.join(OUT, f"grid_{name}_gs{a.gs:g}.png"))
    q = {"n_per_class": a.n_gen, "gs": a.gs, "sampling_minutes": (time.time() - t0) / 60,
         "intensity": {nm: istats(G[L == c]) for c, nm in ((0, "benign"), (1, "malignant"))},
         "intensity_real_val": {nm: istats(XV[YV == c]) for c, nm in ((0, "benign"), (1, "malignant"))}}
    q["saturation_max_class"] = max(q["intensity"][c]["saturation_share"] for c in ("benign", "malignant"))
    del unet; torch.cuda.empty_cache()
    mam = build(); mam.load_state_dict({k: v.float() for k, v in torch.load(find("a6_clf_s0.pt")[0], map_location="cpu").items()})
    mam = mam.to(dev).eval()
    @torch.no_grad()
    def feats(X_u8, bs=64):
        out = []
        for i in range(0, len(X_u8), bs):
            xb = torch.from_numpy(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
            with torch.autocast("cuda", dtype=torch.float16):
                h = mam.classifier[1](mam.classifier[0](mam.avgpool(mam.features(prep(xb)))))
            out.append(h.float().cpu())
        return torch.cat(out).numpy().astype(np.float64)
    ffit = feats(XF)
    q["kid_mammo_vs_trainfit"] = kid(feats(G), ffit)
    q["kid_mammo_real_val_vs_trainfit_floor"] = kid(feats(XV), ffit)
    ref = (thumbs(XV) @ thumbs(XF).T).max(1); q99 = float(np.quantile(ref, 0.99))
    pc = (thumbs(G) @ thumbs(xtr).T).max(1)
    q["memorisation"] = {"reference_pixel_corr_q99": q99, "share_above_q99": float((pc > q99).mean()), "max_pixel_corr": float(pc.max())}
    log(TAG, f"  quality: saturation {q['intensity']['benign']['saturation_share']:.3f}/{q['intensity']['malignant']['saturation_share']:.3f} "
             f"(real val {q['intensity_real_val']['benign']['saturation_share']:.3f}/{q['intensity_real_val']['malignant']['saturation_share']:.3f}) | "
             f"KID-mammo vs train-fit {q['kid_mammo_vs_trainfit'][0]:.4f} (floor {q['kid_mammo_real_val_vs_trainfit_floor'][0]:.4f}) | "
             f"memorisation share {q['memorisation']['share_above_q99']:.3f} | {(time.time() - t0) / 60:.1f} min")
    return q

# ---------------- run ----------------
log(TAG, f"probe set: {a.n_probe} validation images / {a.n_probe} patients, t {GRID}, {a.n_v} probes | validation loss: "
         f"{NV} images x {a.n_rep} | epochs {EPOCHS} | final {a.final}")
for ei, ep in enumerate(EPOCHS):
    path = a.ckpt or os.path.join(a.ckpt_dir, f"{a.arm}_w_ep{ep:03d}.pt")
    if not os.path.exists(path):
        log(TAG, f"!! {path} missing (training not finished?) -> epoch {ep} skipped"); continue
    t_ep = time.time()
    unet, ac, meta = load_model(path, dev)
    LOGAB = torch.log(ac.cpu())
    name = f"{a.arm}_ep{ep:03d}"
    log(TAG, f"epoch {ep}: {path} (snapshot epoch {meta.get('epoch')}, lam {meta.get('lam')})")
    R, A_, P = probe_set(unet, ac, LOGAB)
    Hh = P.mean(1)
    comp, mse = val_loss(unet, ac)
    rep = {"arm": a.arm, "epoch": ep, "checkpoint": path, "lam": meta.get("lam"), "grid": GRID, "n_v": a.n_v,
           "probe_val_index": PIDX.tolist(), "probe_patients": PV[PIDX].tolist(), "probe_labels": LAB,
           "H_mean": float(Hh.mean()), "P_mean_per_t": dict(zip(map(str, GRID), P.mean(0).tolist())),
           "val_composite_mean": float(comp.mean()), "val_mse_mean": float(mse.mean())}
    extra = {}
    if a.final and ei == len(EPOCHS) - 1:
        try:
            rep["jvp_check"] = jvp_check(unet, ac, LOGAB, R)
            log(TAG, f"  JVP check: FD-vs-JVP rel err {rep['jvp_check']['rel_err_fd_vs_jvp']:.4f} | mean P_jvp "
                     f"{np.mean(rep['jvp_check']['P_jvp']):.4f} vs P_fd {np.mean(rep['jvp_check']['P_fd']):.4f}")
        except Exception as e:
            rep["jvp_check"] = {"error": repr(e)[:300]}; log(TAG, f"  !! JVP check failed: {repr(e)[:200]}")
        rep["quality"] = final_quality(unet, ac, name)
    np.savez(os.path.join(OUT, f"eval_{name}.npz"), R=R, A=A_, P=P, H=Hh, val_comp=comp, val_mse=mse, val_t=TV)
    rep["minutes"] = (time.time() - t_ep) / 60
    save_json(rep, f"eval_{name}.json")
    log(TAG, f"epoch {ep}: H mean {Hh.mean():.4f} | P per t " + " ".join(f"{t}:{v:.3f}" for t, v in zip(GRID, P.mean(0)))
             + f" | val loss {comp.mean():.5f} mse {mse.mean():.5f} | {rep['minutes']:.1f} min")
    del unet; torch.cuda.empty_cache()
''',
         "s3_decide.py": r'''# s3_decide.py - S3 decision (pre-stated; VALIDATION images only, the test set is never used):
#   R(lam) = 1 - mean_i H_i(lam) / mean_i H_i(M0) at the final epoch, paired per probe image (same x_t, u, v), 95 % CI from
#   2000 bootstrap resamples of the 32 probe images (= 32 different patients, so image = patient bootstrap).
#   lam is ELIGIBLE if ALL of:
#     R >= 0.10 and CI lower bound > 0                          (the FP term really lowers the FP residual)
#     R_jvp > 0                                                 (the drop is also seen with exact JVP derivatives,
#                                                                i.e. training did not just exploit the FD error)
#     validation training loss <= 1.02 x M0 and validation MSE <= 1.02 x M0
#     saturation share (max over classes, guidance 2) <= 1.25 x M0 (absolute tolerance 0.005) and <= 0.10
#        (the 0.10 cap is dropped, with a flag, if M0 itself is above 0.10)
#     KID-mammo vs train-fit <= 1.5 x M0 (if KID_M0 <= 0: <= KID_M0 + its subset SD)
#   DECISION: the SMALLEST eligible lam -> LAMBDA_SELECTED (goes into the LOCK for Phase C)
#             else, if any lam has CI lower bound > 0 -> INCONCLUSIVE (S3b: extend those arms by 2 epochs)
#             else -> H1_FAIL (the FP term does not measurably lower the residual at these lam -> stop, report)
import os, sys, json, glob, argparse
import numpy as np

ap = argparse.ArgumentParser()
ap.add_argument("--out", default="/kaggle/working/S3")
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
ap.add_argument("--arms", required=True)                  # "s3_m0:0,s3_l016:0.016,..." control (lam = 0) first
ap.add_argument("--final_epoch", type=int, default=42)
ap.add_argument("--epochs", default="41,42")
ap.add_argument("--base_name", default="base")
a = ap.parse_args()
ARMS = [(s.split(":")[0], float(s.split(":")[1])) for s in a.arms.split(",")]
CTRL = ARMS[0][0]; EPOCHS = [int(v) for v in a.epochs.split(",")]
L = []
P_ = lambda *m: L.append(" ".join(str(x) for x in m))

def load(arm, ep):
    f = os.path.join(a.out, f"eval_{arm}_ep{ep:03d}")
    if not (os.path.exists(f + ".npz") and os.path.exists(f + ".json")):
        return None
    return dict(np.load(f + ".npz")), json.load(open(f + ".json"))

def boot_R(Hl, Hm, n=2000, seed=0):
    r = np.random.default_rng(seed); idx = r.integers(0, len(Hl), (n, len(Hl)))
    num, den = Hl[idx].mean(1), Hm[idx].mean(1)
    Rb = 1 - num / np.where(np.abs(den) > 0, den, np.nan)
    return [float(np.nanquantile(Rb, 0.025)), float(np.nanquantile(Rb, 0.975))]

def boot_ratio(xl, xm, n=2000, seed=1):
    r = np.random.default_rng(seed); idx = r.integers(0, len(xl), (n, len(xl)))
    q = xl[idx].mean(1) / xm[idx].mean(1)
    return [float(np.quantile(q, 0.025)), float(np.quantile(q, 0.975))]

def fp_log(arm):
    f = os.path.join(a.ckpt_dir, f"{arm}_fp_log.jsonl")
    if not os.path.exists(f):
        return None
    recs = [json.loads(l) for l in open(f) if l.strip()]
    recs = [r for r in recs if r["epoch"] in EPOCHS]
    if not recs:
        return None
    g = lambda k: np.array([r[k] for r in recs if r.get(k) is not None], float)
    out = {"optimizer_steps": len(recs), "median_base_norm": float(np.median(g("base_norm"))),
           "median_combined_norm": float(np.median(g("combined_norm"))),
           "clipped_share": float((g("combined_norm") > 1.0).mean()), "median_clip_factor": float(np.median(g("clip_factor"))),
           "mean_update_norm": float(g("update_norm").mean()), "mean_pred_dL_base": float(g("pred_dL_base").mean())}
    if len(g("fp_norm_weighted")):
        fpn, bn = g("fp_norm_weighted"), np.array([r["base_norm"] for r in recs if r.get("fp_norm_weighted") is not None], float)
        out.update({"median_fp_norm_weighted": float(np.median(fpn)), "median_fp_over_base": float(np.median(fpn / bn)),
                    "mean_cos_base_fp": float(np.nanmean(g("cos_base_fp"))),
                    "mean_pred_dL_fp_norm": float(g("pred_dL_fp_norm").mean()),
                    "share_steps_FP_predicted_to_decrease": float((g("pred_dL_fp_norm") < 0).mean()),
                    "fp_skipped": int(sum(1 for r in recs if r.get("fp_ok") is False)),
                    "mean_L_fp_norm_per_epoch": {str(e): float(np.mean([r["L_fp_norm"] for r in recs if r["epoch"] == e and r.get("fp_ok")]))
                                                  for e in EPOCHS if any(r["epoch"] == e and r.get("fp_ok") for r in recs)}})
    return out

P_("=" * 110); P_("S3 REPORT - Fokker-Planck feasibility and lambda selection (validation images only; TEST SET NOT USED)"); P_("=" * 110)
P_(f"arms: {ARMS} | control {CTRL} | decision epoch {a.final_epoch}")
ctrl = load(CTRL, a.final_epoch)
res = {"arms": {}, "control": CTRL, "final_epoch": a.final_epoch}
base = load(a.base_name, 40)

# ---------------- residual trajectories ----------------
P_("\n[1] FP RESIDUAL ON THE PROBE SET  (H = mean over t of r1 r2 / S_ref^2; lower = more FP-consistent)")
if base is not None:
    P_(f"  epoch 40 (shared start): H {base[0]['H'].mean():.4f} | per t " + " ".join(f"{t}:{v:.3f}" for t, v in base[1]["P_mean_per_t"].items()))
for arm, lam in ARMS:
    row = []
    for ep in EPOCHS:
        d = load(arm, ep)
        row.append(f"ep{ep} H {d[0]['H'].mean():.4f}" if d else f"ep{ep} missing")
    P_(f"  {arm:10s} (lam {lam:g}): " + " | ".join(row))
if ctrl is None:
    P_("\n!! control arm missing at the decision epoch -> no decision possible (see logs)")
    res["decision"] = "NO_DECISION_MISSING_CONTROL"
else:
    Hm = ctrl[0]["H"]; qm = ctrl[1].get("quality", {}); jm = ctrl[1].get("jvp_check", {})
    P_(f"\n  control {CTRL} epoch {a.final_epoch}: H {Hm.mean():.4f} | per t " + " ".join(f"{t}:{v:.3f}" for t, v in ctrl[1]["P_mean_per_t"].items()))
    if Hm.mean() <= 0:
        P_("  !! control H is not positive -> the probe set cannot resolve the residual; R is undefined")
    sat_m = qm.get("saturation_max_class"); kid_m = qm.get("kid_mammo_vs_trainfit")
    pj_m = np.array(jm.get("P_jvp", []), float)
    flags = []
    if sat_m is not None and sat_m > 0.10:
        flags.append(f"control saturation {sat_m:.3f} > 0.10 -> absolute cap dropped")
    P_("\n[2] PER-ARM CRITERIA at the decision epoch (vs control, paired)")
    for arm, lam in ARMS[1:]:
        d = load(arm, a.final_epoch)
        if d is None:
            P_(f"  {arm}: missing at epoch {a.final_epoch}"); res["arms"][arm] = {"lam": lam, "missing": True}; continue
        Hl, J = d[0]["H"], d[1]
        R = float(1 - Hl.mean() / Hm.mean()) if Hm.mean() > 0 else float("nan")
        ci = boot_R(Hl, Hm) if Hm.mean() > 0 else [float("nan")] * 2
        per_t = {t: float(1 - J["P_mean_per_t"][t] / ctrl[1]["P_mean_per_t"][t]) if ctrl[1]["P_mean_per_t"][t] > 0 else None
                 for t in J["P_mean_per_t"]}
        vc = d[0]["val_comp"].mean(1); vcm = ctrl[0]["val_comp"].mean(1)
        vs = d[0]["val_mse"].mean(1); vsm = ctrl[0]["val_mse"].mean(1)
        r_comp, r_mse = float(vc.mean() / vcm.mean()), float(vs.mean() / vsm.mean())
        q = J.get("quality", {}); jl = J.get("jvp_check", {})
        pj_l = np.array(jl.get("P_jvp", []), float)
        R_jvp = float(1 - pj_l.mean() / pj_m.mean()) if len(pj_l) and len(pj_m) == len(pj_l) and pj_m.mean() > 0 else float("nan")
        sat = q.get("saturation_max_class"); kidl = q.get("kid_mammo_vs_trainfit")
        c = {"R": R, "R_ci95": ci, "R_per_t": per_t, "R_jvp": R_jvp, "fd_vs_jvp_rel_err": jl.get("rel_err_fd_vs_jvp"),
             "val_loss_ratio": r_comp, "val_loss_ratio_ci95": boot_ratio(vc, vcm), "val_mse_ratio": r_mse,
             "val_mse_ratio_ci95": boot_ratio(vs, vsm), "saturation": sat, "saturation_control": sat_m,
             "kid_mammo": kidl, "kid_mammo_control": kid_m, "memorisation": q.get("memorisation")}
        ok = {"R>=0.10": bool(R >= 0.10), "CI_lo>0": bool(ci[0] > 0), "R_jvp>0": bool(R_jvp > 0),
              "val_loss<=1.02": bool(r_comp <= 1.02), "val_mse<=1.02": bool(r_mse <= 1.02)}
        if sat is not None and sat_m is not None:
            ok["saturation"] = bool(sat <= max(1.25 * sat_m, sat_m + 0.005) and (sat <= 0.10 or sat_m > 0.10))
        else:
            ok["saturation"] = False
        if kidl is not None and kid_m is not None:
            ok["kid_mammo"] = bool(kidl[0] <= 1.5 * kid_m[0]) if kid_m[0] > 0 else bool(kidl[0] <= kid_m[0] + kid_m[1])
        else:
            ok["kid_mammo"] = False
        c["checks"] = ok; c["eligible"] = all(ok.values()); c["lam"] = lam
        c["optimisation"] = fp_log(arm)
        res["arms"][arm] = c
        P_(f"  {arm} (lam {lam:g}): R {R:+.3f} CI {ci[0]:+.3f}..{ci[1]:+.3f} | R_jvp {R_jvp:+.3f} (FD-vs-JVP err "
           f"{c['fd_vs_jvp_rel_err'] if c['fd_vs_jvp_rel_err'] is None else round(c['fd_vs_jvp_rel_err'], 4)}) | val loss x{r_comp:.4f} "
           f"mse x{r_mse:.4f} | saturation {sat if sat is None else round(sat, 4)} (control {sat_m if sat_m is None else round(sat_m, 4)}) | "
           f"KID-mammo {None if kidl is None else round(kidl[0], 4)} (control {None if kid_m is None else round(kid_m[0], 4)})")
        P_(f"      R per t: " + " ".join(f"{t}:{v:+.2f}" if v is not None else f"{t}:n/a" for t, v in per_t.items()))
        P_(f"      checks: {ok} -> ELIGIBLE {c['eligible']}")
    # ---------------- decision ----------------
    elig = sorted([(c["lam"], arm) for arm, c in res["arms"].items() if c.get("eligible")])
    sig = [arm for arm, c in res["arms"].items() if not c.get("missing") and c["R_ci95"][0] > 0]
    if elig:
        res["decision"], res["lambda"], res["selected_arm"] = "LAMBDA_SELECTED", elig[0][0], elig[0][1]
    elif sig:
        res["decision"], res["lambda"] = "INCONCLUSIVE", None
        res["s3b_extend_arms"] = sig
    else:
        res["decision"], res["lambda"] = "H1_FAIL", None
    res["flags"] = flags
    P_("\n[3] DECISION")
    P_(f"  {res['decision']}" + (f" -> lambda = {res['lambda']:g} ({res['selected_arm']}) for the LOCK / Phase C" if res.get("lambda") else "")
       + (f" -> S3b: extend {sig} by 2 epochs" if res["decision"] == "INCONCLUSIVE" else "")
       + (" -> the FP term did not lower the residual measurably at these lambda" if res["decision"] == "H1_FAIL" else ""))
    for f in flags:
        P_(f"  FLAG: {f}")

# ---------------- optimisation diagnostics ----------------
P_("\n[4] OPTIMISATION (training logs of the S3 epochs; norms before the clip at 1.0)")
for arm, lam in ARMS:
    o = fp_log(arm)
    res.setdefault("optimisation", {})[arm] = o
    if o is None:
        P_(f"  {arm}: no log"); continue
    P_(f"  {arm} (lam {lam:g}): {o['optimizer_steps']} steps | median |g_base| {o['median_base_norm']:.3g} | median |g_total| "
       f"{o['median_combined_norm']:.3g} | clipped {o['clipped_share']:.2f} (median factor {o['median_clip_factor']:.3f}) | "
       f"mean |update| {o['mean_update_norm']:.3g} | predicted dL_base per step {o['mean_pred_dL_base']:+.3e}")
    if "median_fp_norm_weighted" in o:
        P_(f"      FP: median |lam g_FP| {o['median_fp_norm_weighted']:.3g} (x{o['median_fp_over_base']:.3g} of base) | mean cos(base, FP) "
           f"{o['mean_cos_base_fp']:+.3f} | predicted dL_FP(norm) per step {o['mean_pred_dL_fp_norm']:+.3e} (decrease in "
           f"{o['share_steps_FP_predicted_to_decrease']:.2f} of steps) | training L_FP per epoch {o['mean_L_fp_norm_per_epoch']} | "
           f"skipped {o['fp_skipped']}")
    hp = os.path.join(a.ckpt_dir, f"{arm}_history.json")
    if os.path.exists(hp):
        for h in json.load(open(hp)):
            P_(f"      epoch {h['epoch']}: train loss {h['loss']:.4f} mse {h['mse']:.4f} | {h['minutes']:.1f} min")
P_("\n" + "=" * 110)
P_("Send S3_report.txt (or S3_results.zip) and the grid_*.png images back. The LOCK document is written after this.")
txt = "\n".join(L)
open(os.path.join(a.out, "S3_report.txt"), "w").write(txt)
json.dump(res, open(os.path.join(a.out, "S3_decision.json"), "w"), indent=1, default=float)
print(txt)
'''}
for n, s in FILES.items():
    Path(n).write_text(s)
Path("/tmp/prepare_clean.py").write_text(r'''# prepare_clean.py - CLEAN patient-level CBIS-DDSM cache (rule B, decided 19 Oct 2026 after the data audit).
#   unit            = one unique image (pixel-identical after the 256x256 resize = the same image), not one CSV row
#   resolution      = PatientName in dicom_info (as before); if missing: series UID of the CSV path -> dicom_info, or
#                     the jpeg folder named by that UID holding exactly one image (recovers the 282 skipped test rows)
#   image check     = native min side >= 1000 px and not a binary mask (a full mammogram), else rejected + reported
#   label conflicts = an image carrying both BENIGN* and MALIGNANT rows -> excluded
#   split (rule B)  = patients that occur in both official train and test CSVs go to TEST (all their images);
#                     an image that occurs in both splits therefore ends up in test; train = everything else
#   no class subsampling (classes are already near-balanced)
# Writes /tmp/train_cache.npz and /tmp/test_cache.npz (x uint8 (N,256,256), y int64) exactly like the old prepare.py,
# plus /kaggle/working/clean_data/ (compressed caches with md5 + patient, manifests, summary) so that EVERY later
# notebook can attach this output and use byte-identical data.
import os, glob, json, shutil, subprocess, hashlib, collections, time
import numpy as np, pandas as pd
from PIL import Image

OUT = "/kaggle/working/clean_data"
os.makedirs(OUT, exist_ok=True)
EXPECTED = {"train": 2336, "test": 663}          # data audit, rule B (before the full-mammogram check)

def to_tmp(src_dir):
    for s in ("train", "test"):
        d = np.load(os.path.join(src_dir, f"{s}_cache.npz"))
        np.savez(f"/tmp/{s}_cache.npz", x=d["x"], y=d["y"])
        print(f"{s}: {d['x'].shape} benign={int((d['y'] == 0).sum())} malignant={int((d['y'] == 1).sum())}")

# ---------- 0. already built in this session, or attached from an earlier session -> reuse (identical bytes) ----------
if os.path.exists(os.path.join(OUT, "clean_summary.json")):
    to_tmp(OUT); print("clean caches already built in this session"); raise SystemExit(0)
prev = sorted(glob.glob("/kaggle/input/**/clean_data/clean_summary.json", recursive=True))
if prev:
    src = os.path.dirname(prev[0])
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(OUT, f))
    to_tmp(OUT)
    print("clean caches REUSED from", src); raise SystemExit(0)

t0 = time.time()
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset")
RAW = os.path.dirname(os.path.dirname(hit[0]))
JPEG = os.path.join(RAW, "jpeg")
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
jp = lambda rel: os.path.join(JPEG, str(rel).replace("CBIS-DDSM/jpeg/", ""))
full = info[info["SeriesDescription"] == "full mammogram images"]
by_name = full.set_index("PatientName")["image_path"].to_dict()
by_uid = full.set_index("SeriesInstanceUID")["image_path"].to_dict()
jpeg_dirs = set(os.listdir(JPEG))

# ---------- 1. every CSV row -> image file ----------
rows, how_n = [], collections.Counter()
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test", ["csv/calc_case_description_test_set.csv", "csv/mass_case_description_test_set.csv"])]:
    for c in csvs:
        for _, r in pd.read_csv(os.path.join(RAW, c)).iterrows():
            parts = str(r["image file path"]).strip().split("/")
            rel, how = by_name.get(parts[0]), "patientname"
            if rel is not None and not os.path.exists(jp(rel)):
                rel = None
            if rel is None:
                for comp in parts[1:3]:
                    if comp in by_uid and os.path.exists(jp(by_uid[comp])):
                        rel, how = by_uid[comp], "series_uid_dicom_info"
                        break
            if rel is None:
                for comp in parts[1:3]:
                    if comp in jpeg_dirs:
                        fs = [f for f in os.listdir(os.path.join(JPEG, comp)) if f.lower().endswith(".jpg")]
                        if len(fs) == 1:
                            rel, how = os.path.join(comp, fs[0]), "uid_folder_single_jpeg"
                            break
            if rel is None:
                how = "unresolved"
            how_n[f"{split}:{how}"] += 1
            rows.append({"split": split, "csv": os.path.basename(c), "patient": str(r["patient_id"]),
                         "label": 1 if r["pathology"] == "MALIGNANT" else 0, "rel": rel, "how": how})
R = pd.DataFrame(rows)
print("row resolution:", dict(how_n), flush=True)
Rk = R[R.rel.notna()].copy()

# ---------- 2. load every unique file once: full-mammogram check + 256x256 array + md5 ----------
arr, md5, native, reject = {}, {}, {}, {}
files = sorted(Rk.rel.unique())
for i, rel in enumerate(files):
    with Image.open(jp(rel)) as im:
        W, H = im.size
        g = im.convert("L")
        native[rel] = (W, H)
        if min(W, H) < 1000:
            reject[rel] = f"small ({W}x{H}) - not a full mammogram"
            continue
        small = np.asarray(g.resize((512, 512)))
        if np.mean((small > 30) & (small < 225)) < 0.02:
            reject[rel] = "binary image - looks like an ROI mask"
            continue
        a = np.array(g.resize((256, 256)), dtype=np.uint8)            # identical resize to the old prepare.py
    arr[rel] = a
    md5[rel] = hashlib.md5(a.tobytes()).hexdigest()
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(files)} files ({(time.time() - t0) / 60:.1f} min)", flush=True)
rej_rows = Rk[Rk.rel.isin(reject)]
Rk = Rk[~Rk.rel.isin(reject)].copy()
Rk["md5"] = Rk.rel.map(md5)

# ---------- 3. one record per image, conflicts, rule-B split ----------
ptr, pte = set(R[(R.split == "train") & R.rel.notna()].patient), set(R[(R.split == "test") & R.rel.notna()].patient)
both = ptr & pte                                    # same definition as the data audit
img = Rk.groupby("md5").agg(rel=("rel", "first"), files=("rel", lambda s: sorted(set(s))),
                            splits=("split", lambda s: sorted(set(s))), patients=("patient", lambda s: sorted(set(s))),
                            labels=("label", lambda s: sorted(set(s))), csvs=("csv", lambda s: sorted(set(s)))).reset_index()
conflict = img.labels.map(len) > 1
clean = img[~conflict].copy()
clean["label"] = clean.labels.map(lambda s: s[0])
touch = clean.patients.map(lambda ps: any(p in both for p in ps))
in_tr = clean.splits.map(lambda s: "train" in s)
in_te = clean.splits.map(lambda s: "test" in s)
clean["final_split"] = np.where(in_te | touch, "test", "train")
clean = clean.sort_values("md5").reset_index(drop=True)
assert not (set(p for ps in clean[clean.final_split == "train"].patients for p in ps) &
            set(p for ps in clean[clean.final_split == "test"].patients for p in ps)), "patient overlap remains!"

summary = {"rule": "B: patients in both official splits -> test; label-conflict images excluded; one record per "
                   "pixel-identical image; full-mammogram check", "decided": "2026-10-19",
           "row_resolution": dict(how_n), "rejected_files": len(reject), "rejected_examples": dict(list(reject.items())[:10]),
           "rejected_rows": int(len(rej_rows)), "unique_images": int(len(img)), "label_conflict_images_excluded": int(conflict.sum()),
           "patients_in_both_official_splits": len(both), "build_minutes": None}
for s in ("train", "test"):
    d = clean[clean.final_split == s]
    x = np.stack([arr[r] for r in d.rel]); y = d.label.to_numpy().astype(np.int64)
    np.savez(f"/tmp/{s}_cache.npz", x=x, y=y)
    np.savez_compressed(os.path.join(OUT, f"{s}_cache.npz"), x=x, y=y, md5=d.md5.to_numpy().astype(str),
                        patient=d.patients.map(lambda p: "|".join(p)).to_numpy().astype(str))
    m = d[["md5", "label", "patients", "files", "csvs", "splits"]].copy()
    m["native_WxH"] = d.rel.map(lambda r: "x".join(map(str, native[r])))
    m.to_csv(os.path.join(OUT, f"manifest_{s}.csv"), index=False)
    pats = set(p for ps in d.patients for p in ps)
    summary[s] = {"images": int(len(d)), "benign": int((y == 0).sum()), "malignant": int((y == 1).sum()),
                  "patients": len(pats), "expected_from_audit": EXPECTED[s], "difference": int(len(d)) - EXPECTED[s]}
    print(f"{s}: {x.shape} benign={int((y == 0).sum())} malignant={int((y == 1).sum())} patients={len(pats)} "
          f"(audit expected {EXPECTED[s]}, difference {int(len(d)) - EXPECTED[s]})", flush=True)
summary["build_minutes"] = (time.time() - t0) / 60
json.dump(summary, open(os.path.join(OUT, "clean_summary.json"), "w"), indent=1, default=str)
print(f"rejected files (full-mammogram check): {len(reject)}", dict(list(reject.items())[:5]))
print("clean caches written to /tmp and", OUT, f"({summary['build_minutes']:.1f} min)")
''')
print("wrote", ", ".join(FILES))
subprocess.run([sys.executable, "/tmp/prepare_clean.py"], check=True)      # reuses clean_data/ from the input (identical bytes)

def need(name, hint):
    h = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    print(f"{name}: {h[0] if h else '!! NOT FOUND'}")
    if not h:
        raise SystemExit(f"!! {name} not found -> {hint}")
    return h[0]
INIT = need("base_clean_s42_last.pt", "Add Input -> clean-data-baseline-retraining (latest version)")
BASE40 = need("base_clean_s42_w_ep040.pt", "Add Input -> clean-data-baseline-retraining (latest version)")
A8 = need("A8_fp_clean.json", "Add Input -> the S2 baseline-audit notebook output")
need("a6_clf_s0.pt", "Add Input -> the S2 baseline-audit notebook output")
sref = json.load(open(A8))["S_ref_table"]
print("S_ref(t) from S2:", {k: float(f"{v:.3g}") for k, v in sref.items()})
r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True)
NGPU = int(r.stdout.split()[-1]) if r.returncode == 0 else 0
print("GPUs visible:", NGPU, "" if NGPU >= 2 else "!! expected 2 (Settings -> Accelerator -> GPU T4 x2)")
CK = "/kaggle/working/ckpt"; os.makedirs(CK, exist_ok=True)
LOGS = "/kaggle/working/S3/logs"; os.makedirs(LOGS, exist_ok=True)
RC = {}

def run_one(name, gpu, argv, est_h, out_dir=None):
    """one job in a subprocess; skipped (rc = 'skipped') if it would pass BUDGET_H (0.4 h kept for the report)"""
    if used_h() + est_h > BUDGET_H - 0.4:
        print(f"--- {name} SKIPPED: {used_h():.2f} h used + {est_h} h estimate would pass the budget", flush=True)
        RC[name] = "skipped"; return
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"; env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["CUDA_VISIBLE_DEVICES"] = str(gpu if NGPU >= 2 else 0)
    if out_dir:
        env["PA_OUT"] = out_dir
    print(f">>> {name} started on GPU {env['CUDA_VISIBLE_DEVICES']} | session {used_h():.2f} h", flush=True)
    with open(f"{LOGS}/{name}.log", "w") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush()
            print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]} | session {used_h():.2f} h" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)

def run_queues(queues):
    """queues = {gpu: [(name, argv, est_h), ...]} -> each GPU runs its list in order, the GPUs in parallel"""
    if NGPU < 2:
        queues = {0: [j for q in queues.values() for j in q]}
    ths = [threading.Thread(target=lambda g=g, q=q: [run_one(n, g, a, e) for n, a, e in q]) for g, q in queues.items()]
    for t in ths: t.start()
    for t in ths: t.join()

def train_job(arm, ck_dir=CK, extra=()):
    return ["train_fp.py", "--arm", arm, "--lam", str(ARMS.get(arm, 0.054)), "--epochs", str(TO_EPOCH), "--init", INIT,
            "--sref_json", A8, "--budget_h", str(BUDGET_H - 0.3), "--ckpt_dir", ck_dir] + list(extra)
def eval_job(arm, extra=()):
    return ["s3_eval.py", "--arm", arm, "--epochs", ",".join(str(e) for e in range(41, TO_EPOCH + 1)), "--ckpt_dir", CK,
            "--final"] + list(extra)
print("Cell 3 done.")

In [ ]:
# ===================== CELL 4 — SMOKE TEST (8 micro-batches of FP training + a tiny evaluation, ~10 min) =====================
SM = "/kaggle/working/smoke"
started = any(os.path.exists(f"{CK}/{arm}_last.pt") or glob.glob(f"/kaggle/input/**/{arm}_last.pt", recursive=True)
              for arm in ARMS)
if RUN_SMOKE and not started:
    os.makedirs(f"{SM}/eval", exist_ok=True)
    run_one("smoke_train", 0, ["train_fp.py", "--arm", "smoke", "--lam", "0.054", "--epochs", "41", "--init", INIT,
                               "--sref_json", A8, "--ckpt_dir", SM, "--smoke_batches", "8", "--log_every", "4"], 0.2)
    if RC.get("smoke_train") == 0:
        run_one("smoke_eval", 0, ["s3_eval.py", "--arm", "smoke", "--epochs", "41", "--ckpt_dir", SM, "--final", "--smoke"],
                0.2, out_dir=f"{SM}/eval")
    lines = [json.loads(l) for l in open(f"{SM}/smoke_fp_log.jsonl")] if os.path.exists(f"{SM}/smoke_fp_log.jsonl") else []
    ok = RC.get("smoke_train") == 0 and RC.get("smoke_eval") == 0 and len(lines) == 2 and all(l.get("fp_ok") for l in lines)
    for l in lines:
        print("smoke optimizer step:", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in l.items()})
    if not ok:
        raise SystemExit("SMOKE TEST FAILED -> nothing long was started. Send Claude this block + S3/logs/smoke_*.log")
    shutil.rmtree(SM, ignore_errors=True)
    print(f"SMOKE TEST PASSED | session {used_h():.2f} h")
else:
    print("smoke test skipped", "(training already started in an earlier session)" if started else "(RUN_SMOKE = False)")

In [ ]:
# ===================== CELL 5 — RUN: two GPU queues in parallel (each arm resumes from its own *_last.pt) =====================
t0 = time.time()
run_queues({
    0: [("train_s3_l016", train_job("s3_l016"), 2.0),
        ("train_s3_l107", train_job("s3_l107"), 2.0),
        ("eval_s3_l107", eval_job("s3_l107"), 0.75)],
    1: [("train_s3_l054", train_job("s3_l054"), 2.0),
        ("train_s3_m0", train_job("s3_m0"), 0.45),
        ("eval_s3_m0", eval_job("s3_m0"), 0.75),
        ("eval_s3_l054", eval_job("s3_l054"), 0.75),
        ("eval_s3_l016", eval_job("s3_l016"), 0.75),
        ("eval_base", ["s3_eval.py", "--arm", "base", "--epochs", "40", "--ckpt", BASE40], 0.25)]})
print(f"===== queues done in {(time.time() - t0) / 3600:.2f} h | return codes {RC} | session {used_h():.2f} h =====")

In [ ]:
# ===================== CELL 6 — DECISION + REPORT + zip =====================
arms_arg = ",".join(f"{k}:{v}" for k, v in ARMS.items())
r = subprocess.run([sys.executable, "s3_decide.py", "--arms", arms_arg, "--final_epoch", str(TO_EPOCH),
                    "--epochs", ",".join(str(e) for e in range(41, TO_EPOCH + 1))], cwd=WD, capture_output=True, text=True)
print(r.stdout[-40000:])
if r.returncode != 0:
    print("!! s3_decide.py failed:", r.stderr[-2000:])
TL = "/kaggle/working/S3/train_logs"; os.makedirs(TL, exist_ok=True)
for f in glob.glob(f"{CK}/*_fp_log.jsonl") + glob.glob(f"{CK}/*_history.json"):
    shutil.copy(f, TL)
json.dump(RC, open("/kaggle/working/S3/return_codes.json", "w"), indent=1)
z = shutil.make_archive("/kaggle/working/S3_results", "zip", "/kaggle/working/S3")
print("return codes:", RC, "| ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | session {used_h():.2f} h")
print("checkpoints kept in /kaggle/working/ckpt (needed for S3b / Phase C):",
      sorted(os.path.basename(f) for f in glob.glob(f"{CK}/*.pt")))